# Day 6 — Advanced Analytics & Risk Metrics

This notebook summarises the advanced analytics performed on top of the Day 1-5 pipeline: Value-at-Risk,
rolling risk-adjusted return, investor cohort behaviour, SIP continuity, a simple fund recommender, and
portfolio sector-concentration risk.

All computations live in standalone scripts (kept consistent with the Day 3/4 pattern) so they can be run
independently or scheduled as part of the ETL pipeline:

- `compute_var_cvar.py` → `data/processed/var_cvar_report.csv`
- `compute_rolling_sharpe.py` → `data/processed/rolling_sharpe_90d.csv`, `reports/rolling_sharpe_chart.png`
- `compute_investor_cohorts.py` → `data/processed/cohort_analysis.csv`
- `compute_sip_continuity.py` → `data/processed/sip_continuity.csv`
- `recommender.py` → console fund recommendations by risk appetite
- `compute_sector_hhi.py` → `data/processed/sector_hhi.csv`, `reports/sector_hhi_chart.png`


### Finding 1: Small-cap funds carry the highest tail risk; liquid funds are near risk-free

At 95% confidence, small-cap schemes post the worst daily VaR — ABSL Small Cap (-2.39%), Axis Small Cap
(-2.33%), and SBI Small Cap Direct (-2.32%) top the list, with CVaR (average loss on the worst 5% of days)
around -3.0%. Liquid funds (ICICI Pru, Kotak, ABSL) sit at the opposite end with VaR near -0.02%, confirming
they behave as near-cash instruments. Category-average VaR increases monotonically: Liquid (-0.02%) →
Gilt / Short Duration (-0.33%) → Large Cap (-1.24%) → Mid Cap (-1.67%) → Small Cap (-2.27%) — a clean risk
ladder that validates the dataset's category design.

*See: `compute_var_cvar.py`, Chart: VaR/CVaR ranking (`data/processed/var_cvar_report.csv`).*


### Finding 2: The 2024 investor cohort dominates invested capital, but the 2025 cohort has a higher average ticket size

Investors whose first transaction fell in 2024 account for 4,803 of 5,000 investors and ₹225.8 Cr in total
SIP+Lumpsum capital deployed (avg ₹4.73 lakh/investor), versus 197 investors and ₹1.9 Cr for the 2025 cohort.
This isn't necessarily a sign of declining acquisition — the transaction data only runs through May 2025, so
the 2025 cohort has had far less time to transact. The 2025 cohort's average SIP ticket size (₹13,505) is
actually ~23% higher than 2024's (₹10,997), a more meaningful read on newer-investor behaviour than raw
cohort volume. Both cohorts favour Equity as their top category, with Axis Small Cap and Axis Midcap as the
top individual fund picks respectively.

*See: `compute_investor_cohorts.py` → `data/processed/cohort_analysis.csv`.*


### Finding 3: Nearly all frequent SIP investors show gaps wider than a monthly cadence — flagged as a data-generation limitation

Among 1,362 investors with 6+ SIP transactions, the average gap between SIPs is ~65 days (vs. an expected
~30 for true monthly SIPs), and 97.8% exceed the 35-day "at-risk" threshold, fairly evenly across every age
group (96–99%). Rather than reflecting a real-world lapsing crisis, this points to the synthetic transaction
generator spacing SIP records roughly bi-monthly per investor instead of strictly monthly — the same kind of
caveat flagged for NAV-return correlation in Day 3 EDA. In production, this metric would need re-validation
against real SIP mandate data before being used to trigger investor outreach or churn-prevention campaigns.

*See: `compute_sip_continuity.py` → `data/processed/sip_continuity.csv`.*


### Finding 4: Equity portfolios lean moderately-to-highly concentrated by sector, led by Axis Bluechip

Sector HHI (Herfindahl-Hirschman Index, summed by sector per fund) averages 2,029 across the 34 equity
schemes with holdings data — solidly in the "moderately concentrated" band, with several funds crossing into
"highly concentrated" (HHI > 2,500). Axis Bluechip Fund is the most concentrated (HHI 2,968), followed by
Mirae Asset Tax Saver (2,550), HDFC Mid-Cap Opportunities (2,532), and UTI Flexi Cap (2,514). At the other
end, UTI Mid Cap (1,240), Kotak Flexicap (1,362), and SBI Bluechip Regular (1,425) are the most sector-
diversified. Concentration doesn't map neatly to market-cap category — both a large-cap fund (Axis Bluechip)
and a mid-cap fund (UTI Mid Cap) appear at opposite extremes.

*See: `compute_sector_hhi.py` → `data/processed/sector_hhi.csv`, Chart: Sector HHI by fund.*


### Finding 5: Risk-adjusted efficiency (Sharpe) varies more within categories than risk-grade labels suggest

Liquid ("Low" risk) funds post by far the highest average Sharpe ratio, expected given their near-zero
volatility, while "High" and "Very High" risk equity funds average a much more modest Sharpe — the single
best performer in each bracket (Kotak Emerging Equity, Sharpe 0.96; SBI Small Cap, Sharpe 0.94) barely
clears 1.0. The 90-day rolling Sharpe for the top-5 scorecard funds also shows this efficiency is unstable
over time, drifting negative for all five funds by the most recent window — a reminder that a fund's
scorecard rank at one point in time doesn't guarantee forward risk-adjusted performance. Practically, this
supports a simple recommender rule: within equity risk grades, Sharpe ratio differentiates fund quality far
more than the risk-grade label alone, so `recommender.py` ranks candidates by Sharpe *within* each investor's
matching `risk_category` rather than treating risk grade as a quality proxy.

*See: `compute_rolling_sharpe.py` → `reports/rolling_sharpe_chart.png`; `recommender.py`.*
